# 02 · Overture places for Hyderabad (CPU)

**What this does:** downloads Overture Maps *places* (businesses and points of interest, each with a name, category, address text and coordinates) for the same Hyderabad box as notebook 01, then uses them for two jobs:

1. **More landmarks.** Overture places that OSM doesn't already have are added to the landmark list, which fills the thin outer areas.
2. **Real address text.** Places whose address text looks usable *and* whose location agrees with OSM become candidates for the real-text test set.

| Output (in `MyDrive/Navigate/data/gazetteer/hyderabad/`) | What is in it |
|---|---|
| `overture_places.parquet` | Every Overture place, flattened, with checks against OSM |
| `landmarks_merged.parquet` | OSM landmarks + new Overture landmarks, duplicates removed, `source` column |
| `real_addresses_overture.parquet` | Verified real address strings with coordinates (test-set candidates) |
| `manifest_02.json` | Counts and the coverage report |

**Needs:** notebook 01's outputs on Drive. **Runtime:** CPU, about 10–15 minutes.

The last cell prints a report of what was added.

## 1. Settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
BBOX = dict(west=78.25, south=17.25, east=78.70, north=17.60)
CITY = 'hyderabad'
ROOT = '/content/drive/MyDrive/Navigate/data'
RAW  = f'{ROOT}/raw/overture'
OUT  = f'{ROOT}/gazetteer/{CITY}'
os.makedirs(RAW, exist_ok=True)
assert os.path.exists(f'{OUT}/buildings.parquet'), 'Run notebook 01 first'

MIN_CONF_LANDMARK = 0.60   # Overture confidence needed to add a place as a landmark
MIN_CONF_ADDRESS  = 0.60   # ...and to use its address text in the real test set
DUP_RADIUS_M      = 100    # an Overture place this close to an OSM landmark with a similar name = duplicate
NAME_SIM          = 85     # name similarity (0-100) that counts as "same name"
NEAR_BUILDING_M   = 25     # a place within this distance of an OSM building is "on a building"

## 2. Install tools

In [ ]:
!pip -q install overturemaps rapidfuzz h3
import geopandas as gpd, pandas as pd, numpy as np, shapely, h3
from rapidfuzz import fuzz
print('ready')

## 3. Download Overture places (cached on Drive)
The `overturemaps` tool picks the latest monthly release and only downloads the Hyderabad box.

In [ ]:
import subprocess
PLACES_RAW = f'{RAW}/{CITY}_places.parquet'
if not os.path.exists(PLACES_RAW):
    b = BBOX
    cmd = (f'overturemaps download --bbox={b["west"]},{b["south"]},{b["east"]},{b["north"]} '
           f'-f geoparquet -t place -o "{PLACES_RAW}"')
    print('$', cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    assert os.path.exists(PLACES_RAW) and os.path.getsize(PLACES_RAW) > 10_000, 'Download failed - read the message above'
rel = subprocess.run('overturemaps releases latest', shell=True, capture_output=True, text=True).stdout.strip()
print('Overture release:', rel or '(could not read; see docs.overturemaps.org/release/latest)')
raw = gpd.read_parquet(PLACES_RAW)
print(f'{len(raw):,} places downloaded; columns: {list(raw.columns)}')

## 4. Flatten into simple columns
Overture stores names, categories and addresses as nested fields. This pulls out the parts we need and copes with older and newer schema versions.

In [ ]:
def g(d, *keys):
    for k in keys:
        if d is None: return None
        if isinstance(d, dict): d = d.get(k)
        else:
            try: d = d[k]
            except Exception: return None
    return d

def first(lst):
    if lst is None: return None
    try: return lst[0] if len(lst) else None
    except Exception: return None

def category(row):
    for path in (('basic_category',), ('taxonomy','primary'), ('categories','primary')):
        v = g(row, *path) if len(path) > 1 else row.get(path[0])
        if isinstance(v, str) and v: return v
    return None

rows = []
for rec in raw.drop(columns='geometry').to_dict('records'):
    a = first(rec.get('addresses')) or {}
    src = rec.get('sources')
    rows.append({
        'overture_id': rec.get('id'),
        'name': g(rec, 'names', 'primary'),
        'category': category(rec),
        'confidence': rec.get('confidence'),
        'brand': g(rec, 'brand', 'names', 'primary'),
        'addr_freeform': g(a, 'freeform'),
        'addr_locality': g(a, 'locality'),
        'addr_postcode': g(a, 'postcode'),
        'addr_region': g(a, 'region'),
        'n_sources': len(src) if src is not None else 0,
        'source_datasets': ','.join(sorted({str(g(s, 'dataset')) for s in src})) if src is not None else None,
    })
pl = gpd.GeoDataFrame(rows, geometry=raw.geometry.values, crs='EPSG:4326')
pl['geometry'] = pl.geometry.representative_point()
pl['lat'], pl['lon'] = pl.geometry.y, pl.geometry.x
pl['addr_postcode'] = pl['addr_postcode'].astype('string').str.extract(r'(5\d{5})', expand=False)   # Hyderabad pincodes start with 5
print(pl[['name','category','confidence','addr_freeform','addr_locality','addr_postcode']].head(10).to_string())

## 5. What did we get? (quick exploration)

In [ ]:
eda = {
    'places': len(pl),
    'named': int(pl['name'].notna().sum()),
    'with_category': int(pl['category'].notna().sum()),
    'confidence_median': round(float(pl['confidence'].median()), 3),
    'confidence_ge_0.6': int((pl['confidence'] >= 0.6).sum()),
    'with_freeform_address': int(pl['addr_freeform'].notna().sum()),
    'with_postcode': int(pl['addr_postcode'].notna().sum()),
    'distinct_postcodes': int(pl['addr_postcode'].nunique()),
    'with_locality': int(pl['addr_locality'].notna().sum()),
}
print(json.dumps(eda, indent=2))
print('\nTop categories:\n', pl['category'].value_counts().head(20).to_string())
print('\nSource datasets:\n', pl['source_datasets'].value_counts().head(8).to_string())
print('\nRandom address examples:')
for t in pl['addr_freeform'].dropna().sample(min(15, pl['addr_freeform'].notna().sum()), random_state=1): print('  ', t)

## 6. Check each place against OSM
- **On a building:** is the point within 25 m of an OSM building footprint?
- **Matches an OSM landmark:** is there an OSM landmark within 100 m with a similar name? If yes, it is a duplicate for landmark purposes, and strong evidence the location is right.

In [ ]:
UTM = 'EPSG:32644'
osm_lm = gpd.read_parquet(f'{OUT}/landmarks.parquet')
bld    = gpd.read_parquet(f'{OUT}/buildings.parquet', columns=['osm_id','geometry'])

pl_u, lm_u = pl.to_crs(UTM), osm_lm.to_crs(UTM)

bt = shapely.STRtree(bld.to_crs(UTM).geometry.values)
(i_pl, i_b), d_b = bt.query_nearest(pl_u.geometry.values, max_distance=NEAR_BUILDING_M, return_distance=True, all_matches=False)
pl['near_building'] = False
pl.loc[i_pl, 'near_building'] = True
del bld, bt

def norm(s): return ' '.join(str(s).lower().replace('.', ' ').split()) if isinstance(s, str) else ''
lt = shapely.STRtree(lm_u.geometry.values)
pairs = lt.query(pl_u.geometry.values, predicate='dwithin', distance=DUP_RADIUS_M)
best = {}
for p, l in zip(*pairs):
    s = fuzz.token_set_ratio(norm(pl.at[p, 'name']), norm(osm_lm.at[l, 'name']))
    if s >= NAME_SIM and s > best.get(p, (0, None))[0]: best[p] = (s, l)
pl['osm_match_id'] = None
for p, (s, l) in best.items(): pl.at[p, 'osm_match_id'] = osm_lm.at[l, 'osm_id']
pl['osm_name_match'] = pl['osm_match_id'].notna()
print(f"on a building: {pl.near_building.mean():.1%} | matches an OSM landmark: {pl.osm_name_match.mean():.1%}")

## 7. Merged landmark list
OSM landmarks stay as they are. Overture places are added when they are named, confident enough, and not already in OSM.

In [ ]:
new = pl[pl['name'].notna() & (pl['confidence'] >= MIN_CONF_LANDMARK) & ~pl['osm_name_match']].copy()
new = new.drop_duplicates(subset=['name', 'lat', 'lon'])
new_lm = gpd.GeoDataFrame({
    'landmark_id': 'ov_' + new['overture_id'].astype(str),
    'name': new['name'], 'category': 'overture:' + new['category'].fillna('other'),
    'source': 'overture', 'confidence': new['confidence'],
}, geometry=new.geometry.values, crs='EPSG:4326')
old_lm = gpd.GeoDataFrame({
    'landmark_id': 'osm_' + osm_lm['osm_id'].astype(str),
    'name': osm_lm['name'], 'category': osm_lm['category'],
    'source': 'osm', 'confidence': 1.0,
}, geometry=osm_lm.geometry.values, crs='EPSG:4326')
landmarks_merged = pd.concat([old_lm, new_lm], ignore_index=True)
landmarks_merged['lat'], landmarks_merged['lon'] = landmarks_merged.geometry.y, landmarks_merged.geometry.x
print(f'OSM {len(old_lm):,} + Overture {len(new_lm):,} = {len(landmarks_merged):,} landmarks')

## 8. Real-address candidates
Kept only if: the address text is long enough to be a real address, confidence is high enough, and the location is backed up by OSM (on a building, or name-matched to an OSM landmark).

In [ ]:
def full_text(r):
    parts = [r['addr_freeform'], r['addr_locality'], r['addr_postcode']]
    out = []
    for p in parts:
        if isinstance(p, str) and p.strip() and p.strip().lower() not in ' '.join(out).lower(): out.append(p.strip())
    return ', '.join(out)

cand = pl[pl['addr_freeform'].notna()].copy()
cand['addr_text'] = cand.apply(full_text, axis=1)
cand = cand[cand['addr_text'].str.len() >= 15]
cand = cand[cand['confidence'] >= MIN_CONF_ADDRESS]
cand['verified_by'] = np.select([cand['osm_name_match'], cand['near_building']], ['osm_name_match', 'near_building'], 'none')
real = cand[cand['verified_by'] != 'none'].drop_duplicates(subset=['addr_text']).copy()
real['h3_7'] = [h3.latlng_to_cell(a, o, 7) for a, o in zip(real.lat, real.lon)]
real = real[['overture_id','name','category','addr_text','addr_freeform','addr_locality','addr_postcode',
             'lat','lon','h3_7','confidence','verified_by','source_datasets','geometry']]
print(f'{len(real):,} verified real-address candidates  ({real.verified_by.value_counts().to_dict()})')
for t in real['addr_text'].sample(min(15, len(real)), random_state=2): print('  ', t)

## 9. Coverage before vs after
Same question as notebook 01: share of buildings with a named landmark within 300 m, by distance from the centre.

In [ ]:
b = pd.read_parquet(f'{OUT}/buildings.parquet', columns=['lat','lon'])
bp = gpd.GeoSeries(gpd.points_from_xy(b.lon, b.lat), crs='EPSG:4326').to_crs(UTM).values
def within300(lm):
    t = shapely.STRtree(gpd.GeoSeries(lm.geometry.values, crs='EPSG:4326').to_crs(UTM).values)
    (i, _), _ = t.query_nearest(bp, max_distance=300, return_distance=True, all_matches=False)
    hit = np.zeros(len(bp), bool); hit[i] = True; return hit
b['osm_only'] = within300(old_lm)
b['merged']   = within300(landmarks_merged)
b['ring_km']  = pd.cut(np.hypot((b.lat-17.385)*111, (b.lon-78.4867)*106), [0,5,10,15,20,40])
cov = b.groupby('ring_km', observed=True)[['osm_only','merged']].mean().round(3)
cov['buildings'] = b.groupby('ring_km', observed=True).size()
print(cov.to_string())
print(f"\nWhole city: {b.osm_only.mean():.1%} -> {b.merged.mean():.1%}")

## 10. Save

In [ ]:
def save(gdf, name):
    gdf = gdf.copy()
    for c in gdf.columns:
        if c != 'geometry' and gdf[c].dtype == object: gdf[c] = gdf[c].astype('string')
    gdf.to_parquet(f'{OUT}/{name}.parquet', index=False)
    print(f'{name:28s} {len(gdf):>8,} rows')

save(pl, 'overture_places'); save(landmarks_merged, 'landmarks_merged'); save(real, 'real_addresses_overture')
report = {**eda,
          'share_on_building': round(float(pl.near_building.mean()), 3),
          'share_osm_name_match': round(float(pl.osm_name_match.mean()), 3),
          'new_landmarks_from_overture': len(new_lm),
          'landmarks_merged': len(landmarks_merged),
          'real_address_candidates': len(real),
          'coverage_300m_osm_only': round(float(b.osm_only.mean()), 3),
          'coverage_300m_merged': round(float(b.merged.mean()), 3)}
manifest = {'city': CITY, 'bbox': BBOX, 'created': time.strftime('%Y-%m-%d %H:%M'),
            'source': 'Overture Maps Foundation, places theme', 'release': rel,
            'licence_note': 'Check the places licence for this release on docs.overturemaps.org before publishing',
            'thresholds': dict(MIN_CONF_LANDMARK=MIN_CONF_LANDMARK, MIN_CONF_ADDRESS=MIN_CONF_ADDRESS,
                               DUP_RADIUS_M=DUP_RADIUS_M, NAME_SIM=NAME_SIM, NEAR_BUILDING_M=NEAR_BUILDING_M),
            'report': report}
json.dump(manifest, open(f'{OUT}/manifest_02.json', 'w'), indent=2)
print(json.dumps(report, indent=2))
print('Done.')